# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# 0.1 Mount Google Drive để lưu checkpoint/adapter đề phòng mất session
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 0.2 Đồng bộ mã nguồn repo nếu chạy trên Colab
import os
REPO_DIR = "/content/K4-Track-4-Day10-VisionLanguageModel"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/awun0105/K4-Track-4-Day10-VisionLanguageModel.git /content/K4-Track-4-Day10-VisionLanguageModel
%cd /content/K4-Track-4-Day10-VisionLanguageModel

In [ ]:
# 0.3 Kiểm tra T4 GPU và cài đặt thư viện
# Lưu ý: Chọn menu Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# 0.4 Cấu hình thư mục lưu checkpoint trên Google Drive
import shutil
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/AI20K_Lab_Day10_Checkpoints"
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

def backup_to_drive(src_path, exp_name):
    """Hàm tiện ích sao lưu checkpoint/adapter sang Google Drive"""
    dest = os.path.join(DRIVE_BACKUP_DIR, exp_name)
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dest, dirs_exist_ok=True)
    elif os.path.isfile(src_path):
        shutil.copy2(src_path, dest)
    print(f"✅ Đã sao lưu {src_path} -> {dest}")

In [ ]:
# 0.5 Kaggle & Hugging Face credentials: nạp từ Colab Secrets hoặc file .env / biến môi trường
# TUÂN THỦ QUY ĐỊNH: Tuyệt đối KHÔNG ghi cứng token / secret key vào code.
import os

# 1. Nạp từ Colab Secrets (khi chạy trên Google Colab)
try:
    from google.colab import userdata
    for key in ["KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN"]:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass
except ImportError:
    pass

# 2. Nạp từ file .env (khi chạy local / Google Drive mount có .env)
env_file = ".env"
if os.path.exists(env_file):
    try:
        with open(env_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    k, v = line.split("=", 1)
                    k, v = k.strip(), v.strip().split("#")[0].strip()
                    if k and v and k not in os.environ:
                        os.environ[k] = v
    except Exception:
        pass

# Đồng bộ token cho Hugging Face Hub
if os.environ.get("HF_TOKEN"):
    os.environ["HUGGINGFACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
    try:
        from huggingface_hub import login
        login(token=os.environ["HF_TOKEN"])
        print("✅ Đã xác thực Hugging Face Hub thành công.")
    except Exception as e:
        print(f"Lưu ý Hugging Face login: {e}")

print("Kaggle Username:", "Đã thiết lập" if os.environ.get("KAGGLE_USERNAME") else "Chưa có (cần đặt trong Colab Secrets)")
print("Kaggle Key:", "Đã thiết lập" if os.environ.get("KAGGLE_KEY") else "Chưa có (cần đặt trong Colab Secrets)")
print("HF Token:", "Đã thiết lập" if os.environ.get("HF_TOKEN") else "Chưa có (tuỳ chọn)")


## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# YOUR CODE HERE: đọc data/splits/train.jsonl, xem vài mẫu, vẽ box lên ảnh
import json
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

with open("data/splits/train.jsonl", "r", encoding="utf-8") as f:
    samples = [json.loads(line) for line in f][:3]

for item in samples:
    img = Image.open(f"data/{item['image_path']}")
    draw = ImageDraw.Draw(img)
    w, h = item["width"], item["height"]
    for obj in item["objects"]:
        xc, yc, bw, bh = obj["yolo"]
        x1, y1 = (xc - bw/2) * w, (yc - bh/2) * h
        x2, y2 = (xc + bw/2) * w, (yc + bh/2) * h
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
    plt.figure(figsize=(6, 4))
    plt.imshow(img)
    plt.title(f"Image ID: {item['image_id']} | Objects: {len(item['objects'])}")
    plt.axis("off")
    plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# YOUR CODE HERE: viết yolo_to_box() và objects_to_answer() (GUIDE bước 2)
import json


def yolo_to_box(label: str, yolo: list[float]) -> dict:
    """
    Chuyển đổi 1 nhãn YOLO [xc, yc, w, h] chuẩn hóa 0..1
    sang box Qwen Grounding [x1, y1, x2, y2] trên thang nguyên 0..1000.
    """
    xc, yc, w, h = yolo
    x1 = round((xc - w / 2.0) * 1000.0)
    y1 = round((yc - h / 2.0) * 1000.0)
    x2 = round((xc + w / 2.0) * 1000.0)
    y2 = round((yc + h / 2.0) * 1000.0)

    # Kẹp giá trị (clamp) vào đoạn [0, 1000] để chống tràn viền
    x1 = max(0, min(1000, x1))
    y1 = max(0, min(1000, y1))
    x2 = max(0, min(1000, x2))
    y2 = max(0, min(1000, y2))

    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}


def objects_to_answer(objects: list[dict]) -> str:
    """
    Chuyển danh sách objects thành chuỗi JSON chuẩn.
    Sắp xếp các box theo thứ tự không gian: từ trên xuống dưới (y1), từ trái sang phải (x1).
    Điều này giúp quá trình học ngôn ngữ của LLM ổn định và hội tụ nhanh hơn rất nhiều.
    """
    boxes = [yolo_to_box(obj["label"], obj["yolo"]) for obj in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"), ensure_ascii=False)

In [ ]:
SYSTEM_PROMPT = (
  "You are a helpful road inspection assistant. Detect all road damage objects in the image. "
  "Output a JSON list containing detected items with 'label' and 'bbox_2d'. "
  "Possible labels: 'pothole', 'crack', 'manhole'. "
  "Coordinates must be integers scaled from 0 to 1000 representing [x1, y1, x2, y2]."
)

## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)
import torch
from transformers import (
  AutoProcessor,
  Qwen3VLForConditionalGeneration,
  BitsAndBytesConfig,
)
from pathlib import Path
from tqdm import tqdm

# Cấu hình lượng tử hóa 4-bit NF4
# QUAN TRỌNG: Bỏ qua "visual" và "lm_head" để giữ ở fp16 phục vụ cho E2/E3
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_compute_dtype=torch.float16,
  bnb_4bit_use_double_quant=True,
  llm_int8_skip_modules=["visual", "lm_head"],
)

model_id = "Qwen/Qwen3-VL-4B-Instruct"
processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "left"

model = Qwen3VLForConditionalGeneration.from_pretrained(
  model_id,
  quantization_config=bnb_config,
  torch_dtype=torch.float16,
  device_map={"": 0},
)
model.eval()

# Suy luận trên tập test
import json

with open("data/splits/test.jsonl", "r", encoding="utf-8") as f:
  test_entries = [json.loads(line) for line in f]

Path("work").mkdir(parents=True, exist_ok=True)
out_file = open("work/e0.jsonl", "w", encoding="utf-8")

for item in tqdm(test_entries, desc="Evaluating E0 Zero-shot"):
  img_path = Path("data") / item["image_path"]
  image = Image.open(img_path).convert("RGB")

  messages = [
      {
          "role": "user",
          "content": [
              {"type": "image", "image": image},
              {"type": "text", "text": SYSTEM_PROMPT},
          ],
      }
  ]
  inputs = processor.apply_chat_template(
      messages,
      tokenize=True,
      add_generation_prompt=True,
      return_dict=True,
      return_tensors="pt",
  ).to("cuda")

  prompt_len = inputs["input_ids"].shape[1]
  with torch.no_grad():
      generated_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)

  response = processor.decode(generated_ids[0, prompt_len:], skip_special_tokens=True)
  out_file.write(
      json.dumps({"image_id": item["image_id"], "raw": response}, ensure_ascii=False)
      + "\n"
  )

out_file.close()

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)
from torch.utils.data import Dataset


class RoadDamageDataset(Dataset):
  def __init__(self, jsonl_path, processor):
    with open(jsonl_path, "r", encoding="utf-8") as f:
        self.entries = [json.loads(line) for line in f]
    self.processor = processor

  def __len__(self):
    return len(self.entries)

  def __getitem__(self, idx):
    item = self.entries[idx]
    image = Image.open(Path("data") / item["image_path"]).convert("RGB")
    answer = objects_to_answer(item["objects"])

    # 1. Encode prompt của User
    user_messages = [
      {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": SYSTEM_PROMPT},
        ],
      }
    ]
    prompt_enc = self.processor.apply_chat_template(
        user_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    prompt_len = prompt_enc["input_ids"].shape[1]

    # 2. Encode toàn bộ hội thoại (User + Assistant answer)
    full_messages = [
      {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": SYSTEM_PROMPT},
        ],
      },
      {"role": "assistant", "content": [{"type": "text", "text": answer}]},
    ]
    full_enc = self.processor.apply_chat_template(
        full_messages,
        add_generation_prompt=False,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    input_ids = full_enc["input_ids"][0]
    # Tạo nhãn labels: sao chép input_ids, nhưng gán -100 cho toàn bộ phần Prompt
    labels = input_ids.clone()
    labels[:prompt_len] = -100

    return {
      "input_ids": input_ids,
      "labels": labels,
      "pixel_values": full_enc.get("pixel_values"),
      "image_grid_thw": full_enc.get("image_grid_thw"),
    }


def collate_fn(batch):
  # Padding input_ids và labels về cùng độ dài lớn nhất trong batch
  input_ids = [b["input_ids"] for b in batch]
  labels = [b["labels"] for b in batch]

  padded_input_ids = torch.nn.utils.rnn.pad_sequence(
    input_ids, batch_first=True, padding_value=processor.tokenizer.pad_token_id
  )
  padded_labels = torch.nn.utils.rnn.pad_sequence(
    labels, batch_first=True, padding_value=-100
  )
  attention_mask = (padded_input_ids != processor.tokenizer.pad_token_id).long()

  out = {
    "input_ids": padded_input_ids,
    "attention_mask": attention_mask,
    "labels": padded_labels,
  }

  # Nối các tensor hình ảnh
  if batch[0]["pixel_values"] is not None:
      out["pixel_values"] = torch.cat([b["pixel_values"] for b in batch], dim=0)
  if batch[0]["image_grid_thw"] is not None:
      out["image_grid_thw"] = torch.cat([b["image_grid_thw"] for b in batch], dim=0)

  return out

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)
import re
import torch
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup: str, lora_r: int = 16, lora_alpha: int = 32):
    """
    Cấu hình LoRA và đóng băng/mở tham số theo từng setup thí nghiệm:
    - E1: LoRA chỉ trên LLM (252 modules: q, k, v, o, gate, up, down_proj trên 36 layers).
    - E2: E1 + mở toàn bộ Projector (visual.merger, visual.deepstack_merger_list) ở float32 (~109M params).
    - E3: E2 + LoRA trên cả ViT (thêm 96 modules: attn.qkv, attn.proj, mlp.linear_fc1, mlp.linear_fc2 trên 24 blocks -> tổng 348 modules).
    """
    # 1. Đóng băng tất cả tham số của mô hình gốc
    for p in model.parameters():
        p.requires_grad = False

    # Kích hoạt input gradients cho gradient checkpointing
    model.enable_input_require_grads()

    # 2. Định nghĩa regex target_modules cho LoRA
    setup_upper = setup.upper()
    if setup_upper in ("E1", "E2"):
        # 36 layers LLM * 7 modules = 252 modules
        target_modules = r".*language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)"
    elif setup_upper == "E3":
        # 252 LLM modules + 24 blocks ViT * 4 modules (96) = 348 modules
        target_modules = (
            r".*(language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)|"
            r"visual\.blocks\.\d+\.(attn\.(qkv|proj)|mlp\.(linear_fc1|linear_fc2)))"
        )
    else:
        raise ValueError(f"Setup không hợp lệ: {setup}. Hỗ trợ: 'E1', 'E2', 'E3'.")

    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=0.05,
        target_modules=target_modules,
        bias="none",
        task_type="CAUSAL_LM"
    )

    peft_model = get_peft_model(model, lora_config)

    # 3. Mở toàn bộ Projector (merger + deepstack) ở float32 nếu là E2 hoặc E3
    if setup_upper in ("E2", "E3"):
        opened_projector_count = 0
        for name, param in peft_model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                # Chuyển sang float32 để chống tràn số (numerical stability) khi tích tụ gradient
                param.data = param.data.to(torch.float32)
                param.requires_grad = True
                opened_projector_count += 1
        print(f"[{setup_upper}] Đã mở {opened_projector_count} tensors của Projector ở float32.")

    # Đếm và in số lượng tham số huấn luyện
    trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in peft_model.parameters())
    print(f"[{setup_upper}] Tham số huấn luyện: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")

    return peft_model


## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)
import os
import time
import json
from pathlib import Path
import torch
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
from tqdm import tqdm

def train_setup(peft_model, setup_name: str, max_steps: int = 300, batch_size: int = 2, grad_accum_steps: int = 4):
    """
    Vòng lặp huấn luyện chuẩn theo GUIDE bước 6:
    - Optimizer AdamW lr=2e-4, weight_decay=0.0
    - Cosine LR schedule với warmup 5%
    - Mixed precision fp16 với GradScaler
    - Gradient accumulation (batch 2 * accum 4 = effective batch 8)
    - Gradient clipping max_norm=1.0
    - Gradient checkpointing (use_reentrant=False)
    - Lưu adapter LoRA và riêng projector weights (cho E2/E3)
    - Ghi metadata huấn luyện vào work/run_{setup}.json
    """
    setup_name = setup_name.upper()
    train_dataset = RoadDamageDataset("data/splits/train.jsonl", processor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

    trainable_parameters = [p for p in peft_model.parameters() if p.requires_grad]
    trainable_count = sum(p.numel() for p in trainable_parameters)

    optimizer = AdamW(trainable_parameters, lr=2e-4, weight_decay=0.0)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(max_steps * 0.05),
        num_training_steps=max_steps
    )
    scaler = GradScaler()

    # Bật gradient checkpointing và tắt use_cache để tiết kiệm VRAM
    peft_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    peft_model.config.use_cache = False
    peft_model.train()

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    start_time = time.time()

    step = 0
    accum_loss = 0.0
    loader_iter = iter(train_loader)

    pbar = tqdm(total=max_steps, desc=f"Huấn luyện {setup_name}")
    while step < max_steps:
        optimizer.zero_grad()
        step_loss = 0.0

        for _ in range(grad_accum_steps):
            try:
                batch = next(loader_iter)
            except StopIteration:
                loader_iter = iter(train_loader)
                batch = next(loader_iter)

            batch = {k: v.to("cuda") if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

            with autocast(dtype=torch.float16):
                outputs = peft_model(**batch)
                loss = outputs.loss / grad_accum_steps

            if not torch.isfinite(loss):
                raise RuntimeError(f"Loss không hữu hạn (NaN/Inf) tại step {step}! Dừng để bảo toàn.")

            scaler.scale(loss).backward()
            step_loss += loss.item() * grad_accum_steps

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_parameters, 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        step += 1
        pbar.update(1)
        pbar.set_postfix({"loss": f"{step_loss:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})

        # Lưu checkpoint trung gian mỗi 100 bước
        if step % 100 == 0:
            chk_path = Path(f"work/{setup_name}_adapter_step{step}")
            peft_model.save_pretrained(chk_path)
            if setup_name in ("E2", "E3"):
                proj_weights = {
                    name: param.data.cpu()
                    for name, param in peft_model.named_parameters()
                    if "visual.merger" in name or "visual.deepstack_merger_list" in name
                }
                torch.save(proj_weights, chk_path / "projector_weights.pt")

    pbar.close()

    train_minutes = (time.time() - start_time) / 60.0
    peak_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0

    # Lưu trọng số chính thức sau khi train xong
    save_path = Path(f"work/{setup_name}_adapter")
    save_path.mkdir(parents=True, exist_ok=True)
    peft_model.save_pretrained(save_path)
    print(f"✅ Đã lưu LoRA adapter vào {save_path}")

    # Nếu là E2 hoặc E3: LƯU RIÊNG PROJECTOR WEIGHTS
    if setup_name in ("E2", "E3"):
        projector_weights = {
            name: param.data.cpu()
            for name, param in peft_model.named_parameters()
            if "visual.merger" in name or "visual.deepstack_merger_list" in name
        }
        torch.save(projector_weights, save_path / "projector_weights.pt")
        print(f"✅ Đã lưu {len(projector_weights)} tensors của Projector vào {save_path}/projector_weights.pt")

    meta_info = {
        "trainable_params": int(trainable_count),
        "peak_vram_gb": round(float(peak_vram_gb), 2),
        "train_minutes": round(float(train_minutes), 2)
    }

    # Lưu metadata run.json
    run_file = Path(f"work/run_{setup_name.lower()}.json")
    with open(run_file, "w", encoding="utf-8") as f:
        json.dump(meta_info, f, indent=2)
    print(f"✅ Đã lưu run metadata vào {run_file}: {meta_info}")

    # Tự động sao lưu lên Google Drive nếu hàm backup_to_drive tồn tại
    if "backup_to_drive" in globals():
        try:
            backup_to_drive(str(save_path), f"{setup_name}_adapter")
        except Exception as e:
            print(f"Không thể sao lưu lên Drive: {e}")

    return meta_info

print("Hàm train_setup đã sẵn sàng!")


## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)
import sys
import subprocess
from pathlib import Path
import json
import torch
from peft import PeftModel
from PIL import Image
from tqdm import tqdm

def run_eval(setup_name: str, meta_info: dict = None, split: str = "test", batch_size: int = 1):
    """
    Thực hiện suy luận và chấm điểm chính thức cho setup (E1, E2, E3):
    1. Nạp mô hình gốc 4-bit (giữ visual và lm_head fp16).
    2. Nạp adapter LoRA từ work/{setup_name}_adapter.
    3. Nếu là E2 hoặc E3: nạp lại trọng số Projector từ projector_weights.pt.
    4. Suy luận trên data/splits/{split}.jsonl và xuất work/{setup_name.lower()}.jsonl.
    5. Gọi scripts/score.py kèm metadata: trainable_params, peak_vram_gb, train_minutes.
    """
    setup_name = setup_name.upper()
    adapter_path = Path(f"work/{setup_name}_adapter")
    if not adapter_path.exists():
        raise FileNotFoundError(f"Chưa tìm thấy adapter tại {adapter_path}. Vui lòng chạy huấn luyện trước!")

    # Nếu meta_info chưa truyền vào, cố gắng đọc từ run_{setup}.json
    if meta_info is None:
        run_file = Path(f"work/run_{setup_name.lower()}.json")
        if run_file.exists():
            with open(run_file, "r", encoding="utf-8") as f:
                meta_info = json.load(f)
            print(f"Đã nạp metadata từ {run_file}: {meta_info}")

    print(f"--- BẮT ĐẦU ĐÁNH GIÁ {setup_name} TRÊN SPLIT '{split}' ---")

    # 1. Nạp model gốc 4-bit
    base_model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map={"": 0}
    )

    # 2. Nạp adapter LoRA
    eval_model = PeftModel.from_pretrained(base_model, str(adapter_path))

    # 3. Nạp lại Projector weights (bắt buộc cho E2 và E3)
    proj_path = adapter_path / "projector_weights.pt"
    if proj_path.exists():
        proj_weights = torch.load(proj_path, map_location="cuda")
        eval_model.load_state_dict(proj_weights, strict=False)
        print(f"✅ Đã nạp lại thành công {len(proj_weights)} tensors Projector từ {proj_path}")
    elif setup_name in ("E2", "E3"):
        print(f"⚠️ Cảnh báo: Không tìm thấy projector_weights.pt cho {setup_name}! Kết quả có thể giống E1.")

    eval_model.eval()

    # 4. Đọc split test/val
    split_file = Path(f"data/splits/{split}.jsonl")
    with open(split_file, "r", encoding="utf-8") as f:
        entries = [json.loads(line) for line in f]

    out_file_path = Path(f"work/{setup_name.lower()}.jsonl")
    out_file_path.parent.mkdir(parents=True, exist_ok=True)

    print(f"Đang sinh dự đoán cho {len(entries)} ảnh...")
    with open(out_file_path, "w", encoding="utf-8") as out_f:
        for item in tqdm(entries, desc=f"Suy luận {setup_name}"):
            img = Image.open(Path("data") / item["image_path"]).convert("RGB")
            messages = [
                {"role": "user", "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": SYSTEM_PROMPT}
                ]}
            ]
            inputs = processor.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt"
            ).to("cuda")

            prompt_len = inputs["input_ids"].shape[1]
            with torch.no_grad():
                out = eval_model.generate(**inputs, max_new_tokens=256, do_sample=False)

            raw = processor.decode(out[0, prompt_len:], skip_special_tokens=True)
            out_f.write(json.dumps({"image_id": item["image_id"], "raw": raw}, ensure_ascii=False) + "\n")

    print(f"✅ Đã lưu kết quả suy luận vào {out_file_path}")

    # Giải phóng GPU VRAM
    del eval_model
    del base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # 5. Chạy chấm điểm chính thức với scripts/score.py
    evidence_out = Path(f"report/evidence/{setup_name.lower()}.json")
    evidence_out.parent.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable, "scripts/score.py",
        "--pred", str(out_file_path),
        "--name", setup_name,
        "--split", split,
        "--out", str(evidence_out)
    ]

    if meta_info:
        for k in ["trainable_params", "peak_vram_gb", "train_minutes"]:
            if k in meta_info:
                cmd.extend(["--meta", f"{k}={meta_info[k]}"])

    print("Chạy câu lệnh chấm điểm:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    print(f"✅ Đã chấm điểm xong {setup_name}! Báo cáo tại: {evidence_out}")

print("Hàm run_eval đã sẵn sàng!")


## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9